## USing videos from: https://www.youtube.com/watch?v=jZMCyqg-85Q&list=PL7W-V4kZrOJNkgPvjXJspyPBxENXj1QX9

In [ ]:
import os
import csv
import json
import re
from datetime import timedelta

import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from skimage.metrics import structural_similarity as ssim
import unicodedata

# ---------------- CONFIG ----------------
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESSERACT_CONFIG = "--psm 6 --oem 3 -l eng -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'-. "

# Folder-specific start/end times (HH:MM:SS)
FOLDER_TIMES = {
    "2026-05-11-graduate": ("00:47:00", "02:01:40"),
    "2026-05-14-graduate": ("00:35:33", "01:43:26"),
}

# Special folders with dual ROI
SPECIAL_FOLDERS = {"2026-05-11-graduate", "2026-05-14-graduate"}

# Fractions for special dual-ROI layout
LEFT_X0_FRAC, LEFT_X1_FRAC = 0.0, 264.0 / 640
RIGHT_X0_FRAC, RIGHT_X1_FRAC = 374.0 / 640, 1.0

# SSIM threshold
SSIM_THRESHOLD = 0.92


# ---------------- HELPERS ----------------
def normalize_folder_name(name):
    return unicodedata.normalize("NFKC", name).strip().lower() if name else ""


def is_ceremony_folder(name):
    return re.match(r"^\d{4}-\d{2}-\d{2}-", normalize_folder_name(name)) is not None


def ts_to_seconds(ts_str):
    parts = ts_str.split(":")
    if len(parts) == 2:
        h = 0
        m, s = parts
    else:
        h, m, s = parts
    return int(h) * 3600 + int(m) * 60 + float(s)


def format_timestamp_seconds(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"


def wrap_text(text, width_px, font_scale, thickness):
    words = text.split(' ')
    lines, current_line = [], []
    for word in words:
        test_line = ' '.join(current_line + [word])
        size = cv2.getTextSize(test_line, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)[0]
        if size[0] <= width_px:
            current_line.append(word)
        else:
            if current_line:
                lines.append(' '.join(current_line))
            current_line = [word]
    if current_line:
        lines.append(' '.join(current_line))
    return lines


def force_spaces(text):
    if not text:
        return ""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-z])(of|the|and|for|in|to)\b", r"\1 \2", text, flags=re.IGNORECASE)
    return " ".join(text.split())


# ---------------- DYNAMIC BOTTOM-ANCHORED ROI ----------------
def get_crops_for_frame(img, folder_name):
    h, w = img.shape[:2]
    base = os.path.basename(folder_name)

    # These values come from the REAL 640x360 lower-third layout
    ROI_TOP_FRAC = 260 / 360
    ROI_BOTTOM_FRAC = 1.0

    roi_top = int(h * ROI_TOP_FRAC)
    roi_bottom = int(h * ROI_BOTTOM_FRAC)

    if base in SPECIAL_FOLDERS:
        # Dual ROI horizontal splits based on 640px layout
        left_x0  = int(w * (0   / 640))
        left_x1  = int(w * (264 / 640))
        right_x0 = int(w * (374 / 640))
        right_x1 = int(w * (640 / 640))

        return [
            (left_x0,  roi_top, left_x1,  roi_bottom, "left"),
            (right_x0, roi_top, right_x1, roi_bottom, "right"),
        ]

    else:
        # Single ROI (full width)
        return [(0, roi_top, w, roi_bottom, "single")]



# ---------------- CORE PIPELINE PER FOLDER ----------------
def run_pipeline_for_folder(folder):
    folder_base = os.path.basename(folder)
    if folder_base not in FOLDER_TIMES:
        print(f"Skipping {folder} (no start/end times configured).")
        return

    print(f"\n=== PROCESSING FOLDER: {folder_base} ===")

    # Find video file
    v_file = next(
        (os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".mp4")),
        None
    )
    if not v_file:
        print("No .mp4 found, skipping.")
        return

    start_str, end_str = FOLDER_TIMES[folder_base]
    start_sec = ts_to_seconds(start_str)
    end_sec = ts_to_seconds(end_str)

    annot_dir = os.path.join(folder, "frames_annotated")
    os.makedirs(annot_dir, exist_ok=True)

    csv_path = os.path.join(folder, "ocr_results.csv")
    json_path = os.path.join(folder, "ocr_results.json")

    # -------- RESUME LOGIC --------
    resume_sec = start_sec
    csv_exists = os.path.exists(csv_path)

    if csv_exists:
        with open(csv_path, "r", encoding="utf-8") as f:
            rows = list(csv.reader(f))
            if len(rows) > 1:
                last_ts = rows[-1][1]
                resume_sec = ts_to_seconds(last_ts)
                print(f"Found existing CSV file: {csv_path}")
                print(f"Resuming last location of: {last_ts}")
            else:
                print(f"CSV exists but empty, starting at {start_str}.")
    else:
        print(f"No CSV found for {folder_base}. Starting at {start_str}.")

    # -------- VIDEO SETUP --------
    cap = cv2.VideoCapture(v_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_MSEC, resume_sec * 1000)

    frame_number = int(resume_sec * fps)

    # Last ROI per side
    last_rois = {}
    last_texts = {}

    # -------- JSON SETUP --------
    json_file = open(json_path, "w", encoding="utf-8")
    json_file.write("[\n")
    first_json = True

    # -------- CSV SETUP --------
    csv_file = open(csv_path, "a", newline="", encoding="utf-8")
    writer = csv.writer(csv_file)
    if not csv_exists:
        writer.writerow(["index", "timestamp", "side", "text"])

    # -------- MAIN LOOP --------
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        current_sec = resume_sec + ((frame_number - int(resume_sec * fps)) / fps)
        if current_sec > end_sec:
            break

        ts_str = format_timestamp_seconds(current_sec)
        crops = get_crops_for_frame(frame, folder_base)

        changed_sides = []

        # Check each ROI independently
        for (x0, y0, x1, y1, side) in crops:
            roi = frame[y0:y1, x0:x1]
            if roi.size == 0:
                continue

            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

            if side in last_rois:
                prev = last_rois[side]
                if roi_gray.shape != prev.shape:
                    roi_gray_resized = cv2.resize(roi_gray, (prev.shape[1], prev.shape[0]))
                    score = ssim(roi_gray_resized, prev)
                else:
                    score = ssim(roi_gray, prev)
                changed = score < SSIM_THRESHOLD
            else:
                changed = True

            if changed:
                changed_sides.append(side)

        if not changed_sides:
            frame_number += 1
            continue

        annotated = frame.copy()

        # Process each changed ROI independently
        for (x0, y0, x1, y1, side) in crops:
            if side not in changed_sides:
                continue

            roi = frame[y0:y1, x0:x1]
            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            last_rois[side] = roi_gray.copy()

            cv2.rectangle(annotated, (x0, y0), (x1, y1), (0, 255, 0), 2)

            text = force_spaces(pytesseract.image_to_string(roi, config=TESSERACT_CONFIG).strip())

            # Prevent duplicate text
            if side in last_texts and last_texts[side] == text:
                continue

            last_texts[side] = text

            # Draw text at top of ROI
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            thickness = 2
            color = (0, 0, 255)
            outline_color = (255, 255, 255)

            lines = wrap_text(f"{side.upper()}: {text}", (x1 - x0), font_scale, thickness)
            for i, line in enumerate(lines):
                y_pos = y0 + 35 + (i * 25)
                cv2.putText(annotated, line, (x0 + 5, y_pos),
                            font, font_scale, outline_color, thickness + 2, cv2.LINE_AA)
                cv2.putText(annotated, line, (x0 + 5, y_pos),
                            font, font_scale, color, thickness, cv2.LINE_AA)

            # Save annotated frame
            safe_ts = ts_str.replace(":", "-")
            out_name = os.path.join(
                annot_dir,
                f"frame_{frame_number:06d}_{side}_{safe_ts}_annotated.jpg"
            )
            cv2.imwrite(out_name, annotated, [int(cv2.IMWRITE_JPEG_QUALITY), 100])

            # CSV entry
            writer.writerow([frame_number, ts_str, side, text])
            csv_file.flush()

            # JSON entry
            entry = {
                "index": str(frame_number),
                "timestamp": ts_str,
                "side": side,
                "text": text,
            }

            if not first_json:
                json_file.write(",\n")
            first_json = False

            json_file.write(json.dumps(entry, indent=2))
            json_file.flush()
            os.fsync(json_file.fileno())

            print(f"[DETECTION] {folder_base} @ {ts_str} — {side.upper()}: \"{text}\"")

        frame_number += 1

    cap.release()

    json_file.write("\n]\n")
    json_file.close()
    csv_file.close()

    print(f"Finished folder: {folder_base}")


# ---------------- RUN ALL FOLDERS ----------------
def run_all_folders():
    cwd = os.getcwd()
    for root, dirs, files in os.walk(cwd):
        for d in dirs:
            if is_ceremony_folder(d):
                folder_path = os.path.join(root, d)
                run_pipeline_for_folder(folder_path)


if __name__ == "__main__":
    run_all_folders()


In [5]:
import os
import csv
import json
import re
from datetime import timedelta

import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from skimage.metrics import structural_similarity as ssim
import unicodedata

# ---------------- CONFIG ----------------
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESSERACT_CONFIG = "--psm 6 --oem 3 -l eng -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'-. "

# Folder-specific start/end times (HH:MM:SS)
FOLDER_TIMES = {
    "2026-05-11-graduate": ("00:47:00", "02:01:40"),
    "2026-05-14-graduate": ("00:35:33", "01:43:26"),
}

# Special folders with dual ROI
SPECIAL_FOLDERS = {"2026-05-11-graduate", "2026-05-14-graduate"}

# Fractions for special dual-ROI layout
LEFT_X0_FRAC, LEFT_X1_FRAC = 0.0, 264.0 / 640
RIGHT_X0_FRAC, RIGHT_X1_FRAC = 374.0 / 640, 1.0

# SSIM threshold
SSIM_THRESHOLD = 0.92


# ---------------- HELPERS ----------------
def normalize_folder_name(name):
    return unicodedata.normalize("NFKC", name).strip().lower() if name else ""


def is_ceremony_folder(name):
    return re.match(r"^\d{4}-\d{2}-\d{2}-", normalize_folder_name(name)) is not None


def ts_to_seconds(ts_str):
    parts = ts_str.split(":")
    if len(parts) == 2:
        h = 0
        m, s = parts
    else:
        h, m, s = parts
    return int(h) * 3600 + int(m) * 60 + float(s)


def format_timestamp_seconds(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"


def wrap_text(text, width_px, font_scale, thickness):
    words = text.split(' ')
    lines, current_line = [], []
    for word in words:
        test_line = ' '.join(current_line + [word])
        size = cv2.getTextSize(test_line, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)[0]
        if size[0] <= width_px:
            current_line.append(word)
        else:
            if current_line:
                lines.append(' '.join(current_line))
            current_line = [word]
    if current_line:
        lines.append(' '.join(current_line))
    return lines


def force_spaces(text):
    if not text:
        return ""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-z])(of|the|and|for|in|to)\b", r"\1 \2", text, flags=re.IGNORECASE)
    return " ".join(text.split())


# ---------------- DYNAMIC BOTTOM-ANCHORED ROI ----------------
def get_crops_for_frame(img, folder_name):
    h, w = img.shape[:2]
    base = os.path.basename(folder_name)

    # These values come from the REAL 640x360 lower-third layout
    ROI_TOP_FRAC = 260 / 360
    ROI_BOTTOM_FRAC = 1.0

    roi_top = int(h * ROI_TOP_FRAC)
    roi_bottom = int(h * ROI_BOTTOM_FRAC)

    if base in SPECIAL_FOLDERS:
        # Dual ROI horizontal splits based on 640px layout
        left_x0  = int(w * (0   / 640))
        left_x1  = int(w * (264 / 640))
        right_x0 = int(w * (374 / 640))
        right_x1 = int(w * (640 / 640))

        return [
            (left_x0,  roi_top, left_x1,  roi_bottom, "left"),
            (right_x0, roi_top, right_x1, roi_bottom, "right"),
        ]

    else:
        # Single ROI (full width)
        return [(0, roi_top, w, roi_bottom, "single")]



# ---------------- CORE PIPELINE PER FOLDER ----------------
def run_pipeline_for_folder(folder):
    folder_base = os.path.basename(folder)
    if folder_base not in FOLDER_TIMES:
        print(f"Skipping {folder} (no start/end times configured).")
        return

    print(f"\n=== PROCESSING FOLDER: {folder_base} ===")

    # Find video file
    v_file = next(
        (os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".mp4")),
        None
    )
    if not v_file:
        print("No .mp4 found, skipping.")
        return

    start_str, end_str = FOLDER_TIMES[folder_base]
    start_sec = ts_to_seconds(start_str)
    end_sec = ts_to_seconds(end_str)

    annot_dir = os.path.join(folder, "frames_annotated")
    os.makedirs(annot_dir, exist_ok=True)

    csv_path = os.path.join(folder, "ocr_results.csv")
    json_path = os.path.join(folder, "ocr_results.json")

    # -------- RESUME LOGIC --------
    resume_sec = start_sec
    csv_exists = os.path.exists(csv_path)

    if csv_exists:
        with open(csv_path, "r", encoding="utf-8") as f:
            rows = list(csv.reader(f))
            if len(rows) > 1:
                last_ts = rows[-1][1]
                resume_sec = ts_to_seconds(last_ts)
                print(f"Found existing CSV file: {csv_path}")
                print(f"Resuming last location of: {last_ts}")
            else:
                print(f"CSV exists but empty, starting at {start_str}.")
    else:
        print(f"No CSV found for {folder_base}. Starting at {start_str}.")

    # -------- VIDEO SETUP --------
    cap = cv2.VideoCapture(v_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_MSEC, resume_sec * 1000)

    frame_number = int(resume_sec * fps)

    # Last ROI per side
    last_rois = {}
    last_texts = {}

    # -------- JSON SETUP --------
    json_file = open(json_path, "w", encoding="utf-8")
    json_file.write("[\n")
    first_json = True

    # -------- CSV SETUP --------
    csv_file = open(csv_path, "a", newline="", encoding="utf-8")
    writer = csv.writer(csv_file)
    if not csv_exists:
        writer.writerow(["index", "timestamp", "side", "text"])

    # -------- MAIN LOOP --------
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        current_sec = resume_sec + ((frame_number - int(resume_sec * fps)) / fps)
        if current_sec > end_sec:
            break

        ts_str = format_timestamp_seconds(current_sec)
        crops = get_crops_for_frame(frame, folder_base)

        changed_sides = []

        # Check each ROI independently
        for (x0, y0, x1, y1, side) in crops:
            roi = frame[y0:y1, x0:x1]
            if roi.size == 0:
                continue

            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

            if side in last_rois:
                prev = last_rois[side]
                if roi_gray.shape != prev.shape:
                    roi_gray_resized = cv2.resize(roi_gray, (prev.shape[1], prev.shape[0]))
                    score = ssim(roi_gray_resized, prev)
                else:
                    score = ssim(roi_gray, prev)
                changed = score < SSIM_THRESHOLD
            else:
                changed = True

            if changed:
                changed_sides.append(side)

        if not changed_sides:
            frame_number += 1
            continue

        annotated = frame.copy()

        # Process each changed ROI independently
        for (x0, y0, x1, y1, side) in crops:
            if side not in changed_sides:
                continue

            roi = frame[y0:y1, x0:x1]
            roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            last_rois[side] = roi_gray.copy()

            cv2.rectangle(annotated, (x0, y0), (x1, y1), (0, 255, 0), 2)

            text = force_spaces(pytesseract.image_to_string(roi, config=TESSERACT_CONFIG).strip())

            # Prevent duplicate text
            if side in last_texts and last_texts[side] == text:
                continue

            last_texts[side] = text

            # Draw text at the absolute top of the frame
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            thickness = 2
            color = (0, 0, 255)
            outline_color = (255, 255, 255)

            lines = wrap_text(f"{side.upper()}: {text}", (x1 - x0), font_scale, thickness)
            for i, line in enumerate(lines):
                y_pos = 35 + (i * 25) # REMOVED y0 so it anchors to the top of the frame
                cv2.putText(annotated, line, (x0 + 5, y_pos),
                            font, font_scale, outline_color, thickness + 2, cv2.LINE_AA)
                cv2.putText(annotated, line, (x0 + 5, y_pos),
                            font, font_scale, color, thickness, cv2.LINE_AA)

            # Save annotated frame
            safe_ts = ts_str.replace(":", "-")
            out_name = os.path.join(
                annot_dir,
                f"frame_{frame_number:06d}_{side}_{safe_ts}_annotated.jpg"
            )
            cv2.imwrite(out_name, annotated, [int(cv2.IMWRITE_JPEG_QUALITY), 100])

            # CSV entry
            writer.writerow([frame_number, ts_str, side, text])
            csv_file.flush()

            # JSON entry
            entry = {
                "index": str(frame_number),
                "timestamp": ts_str,
                "side": side,
                "text": text,
            }

            if not first_json:
                json_file.write(",\n")
            first_json = False

            json_file.write(json.dumps(entry, indent=2))
            json_file.flush()
            os.fsync(json_file.fileno())

            print(f"[DETECTION] {folder_base} @ {ts_str} — {side.upper()}: \"{text}\"")

        frame_number += 1

    cap.release()

    json_file.write("\n]\n")
    json_file.close()
    csv_file.close()

    print(f"Finished folder: {folder_base}")


# ---------------- RUN ALL FOLDERS ----------------
def run_all_folders():
    cwd = os.getcwd()
    for root, dirs, files in os.walk(cwd):
        for d in dirs:
            if is_ceremony_folder(d):
                folder_path = os.path.join(root, d)
                run_pipeline_for_folder(folder_path)


if __name__ == "__main__":
    run_all_folders()


=== PROCESSING FOLDER: 2026-05-11-graduate ===
No CSV found for 2026-05-11-graduate. Starting at 00:47:00.
[DETECTION] 2026-05-11-graduate @ 00:47:00.000 — LEFT: ""
[DETECTION] 2026-05-11-graduate @ 00:47:00.000 — RIGHT: "waawa"
[DETECTION] 2026-05-11-graduate @ 00:47:01.568 — RIGHT: "wa R"
[DETECTION] 2026-05-11-graduate @ 00:47:01.735 — LEFT: "ee"
[DETECTION] 2026-05-11-graduate @ 00:47:01.735 — RIGHT: "Jiann-Ping Hsu College of Public Health"
[DETECTION] 2026-05-11-graduate @ 00:47:05.639 — RIGHT: "Reichley Jude Henderson Cum Laude"
[DETECTION] 2026-05-11-graduate @ 00:47:18.752 — LEFT: "Osazuwa Edw in Unoko"
[DETECTION] 2026-05-11-graduate @ 00:47:21.288 — RIGHT: "Trishell Tiknika Alford"
[DETECTION] 2026-05-11-graduate @ 00:47:26.693 — LEFT: "Ryleigh L.Pierson"
[DETECTION] 2026-05-11-graduate @ 00:47:29.062 — RIGHT: "Beatrice Maigua"
[DETECTION] 2026-05-11-graduate @ 00:47:36.403 — RIGHT: "Aubrie Alaina Kern Magna Cum Laude"
[DETECTION] 2026-05-11-graduate @ 00:47:41.174 — LEFT: 